# Week 10 — Tell the Story (STARTER)
### Week 10 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

---

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability

This ML pipeline works with **any binary classification dataset**.
Change `DATA_FILE`, `TARGET_COL`, and `POSITIVE_LABEL` to use your own data.

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Building the dashboard for the analyst, not the user | If the user needs instructions to operate the dashboard, it has failed. Every control should be self-explanatory. | Test your dashboard with someone who was not involved in building it |
| Saving the model without the preprocessing pipeline | A model saved without its scaler and feature list cannot be used in production — it will break on new data. | Always save model + scaler + feature_columns together as a single pipeline object |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Build a Streamlit dashboard with at least three controls. A non-technical user must be able to answer: what does the model predict, how confident is it, and is it fair?

> 🔬 **Advanced Path**
> Save your complete model pipeline (preprocessing + model) to a pickle file. Write a `predict_new_customer()` function that loads the pipeline and returns a prediction with confidence score.

## ✅ What You Can Do After This Week

- Serialise a model pipeline (model + scaler + feature columns) to a pickle file
- Write a `score_customers()` function that handles data with and without a target column
- Build a four-section Streamlit dashboard with caching, filters, table, and export
- Use `@st.cache_data` to prevent expensive recomputation on every user interaction
- Demo a dashboard to a non-technical stakeholder and answer live questions

*Add the `kova_dashboard/` folder to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week10_storytelling_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week10_storytelling_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **SOLUTION notebook** — One complete working version. Yours does not need to match — there are many correct approaches.

> ⏸️ **Pause and Predict**
> Before building the dashboard: write down the three questions a non-technical operations manager would want to answer using this tool. Then check that your dashboard answers all three without requiring any explanation from you.

In [1]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

📦 Google Colab detected — installing libraries...
✅ Libraries installed.


## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook is written for the Telco Churn dataset but the **entire ML pipeline works on any binary classification problem**.

**To swap in your own data, change only these lines at the top of the notebook:**
```python
DATA_FILE      = 'customer_churn.csv'  # ← your CSV filename
TARGET_COL     = 'Churn'              # ← the binary column you want to predict (Yes/No or 1/0)
POSITIVE_LABEL = 'Yes'                # ← which value means "positive" (the event you care about)
ID_COL         = 'customerID'         # ← identifier column to drop before modelling (or None)
NUMERIC_FIX    = 'TotalCharges'       # ← any numeric column stored as text (or None)
```

**Works with any binary classification dataset:**
- 🏦 Loan default prediction
- 🏥 Patient readmission risk
- 📧 Email spam detection
- 💳 Credit card fraud
- 🛒 Customer conversion
- 🎓 Student dropout prediction

> ⚠️ **Class imbalance warning:** If your dataset has <20% positive class, use `class_weight='balanced'` — already done in this notebook. Check your class distribution in Step 4 before proceeding.

---

## 🔄 Using a Different Dataset?

This notebook is written for `customer_churn.csv` but the ML pipeline applies to any binary classification problem.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Set `TARGET_COL` to your binary target column (the thing you want to predict)
3. Update `POSITIVE_LABEL` to the positive class name (e.g. `'Yes'`, `'1'`, `'Fraud'`)
4. The `prepare_dataset()` function encodes features — update the column references to match yours

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE      = 'customer_churn.csv'   # ← change to your file
TARGET_COL     = 'Churn'               # ← your binary target column
POSITIVE_LABEL = 'Yes'                 # ← the positive class value
ID_COL         = 'customerID'          # ← identifier column to drop (or None)
```

**Works with any binary classification dataset** — fraud detection, disease prediction,
loan default, email spam, customer conversion — as long as your target has two classes.

---

## What This Notebook Does

Trains the final model on the **full dataset** and saves everything needed
for the Kova churn risk dashboard.

**After running all cells, `kova_dashboard/` will contain:**
- `model_pipeline.pkl` — trained model, scaler, and feature column list
- `scored_customers.csv` — all customers scored and ranked

> **Why full dataset?** The train-test split was for evaluation only.
> For deployment we use all available labelled data.
> Real-world performance is measured from the pilot — not from past test scores.

---

In [2]:
import pandas as pd, numpy as np, pickle, os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
os.makedirs('kova_dashboard', exist_ok=True)
print('✅ Libraries loaded | kova_dashboard/ ready')

✅ Libraries loaded | kova_dashboard/ ready


In [3]:
def prepare_dataset(df):
    df=df.copy()
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
    df['TotalCharges']=df['TotalCharges'].fillna(0)  # new customers have zero charges
    if 'customerID' in df.columns: df=df.drop(columns=['customerID'])
    le=LabelEncoder(); df['Churn']=le.fit_transform(df['Churn'].astype(str))
    df['tenure_charge_ratio']=df['TotalCharges']/(df['tenure']+1)
    sc=[c for c in ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                     'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
        if c in df.columns]
    for col in sc: df[f'{col}_binary']=df[col].apply(lambda x:1 if str(x).lower()=='yes' else 0)
    df['num_services']=df[[f'{c}_binary' for c in sc]].sum(axis=1)
    df['contract_numeric']=df['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})
    df['paperless_numeric']=(df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction']=df['contract_numeric']*df['paperless_numeric']
    df['tenure_segment']=pd.cut(df['tenure'],bins=[0,12,36,float('inf')],labels=['New','Established','Loyal'])
    return df
print('✅ prepare_dataset() defined')

✅ prepare_dataset() defined


## Step 1 — Load Data (preserve customerID for display)

In [9]:
import pandas as pd, os

# Smart loader: works locally (from package) and in Colab (from GitHub)
LOCAL_PATH  = '../data/customer_churn.csv'
REMOTE_URL  = 'https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/datasets/customer_churn.csv'
DATA_SOURCE = LOCAL_PATH if os.path.exists(LOCAL_PATH) else REMOTE_URL
df = pd.read_csv(DATA_SOURCE)

# Preserve original customer ID for display later (as per notebook context)
customer_ids = df['customerID']
df_original = df.copy() # Make a copy to preserve original, especially customerID

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns ✅')
print(f'Source: {DATA_SOURCE}')
df.head()

Dataset loaded: 7,043 rows × 21 columns ✅
Source: https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/datasets/customer_churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 2 — Prepare and Encode

In [13]:
df_prep = prepare_dataset(df_original)
cat_cols = [c for c in df_prep.select_dtypes(include='object').columns if c!='Churn']
df_enc   = pd.get_dummies(df_prep, columns=cat_cols, drop_first=True)
# All columns except the target — these are the model inputs
feature_cols = [c for c in df_enc.columns if c!='Churn']
# Feature matrix — everything the model learns from
X = df_enc[feature_cols].copy() # Explicitly create a copy to avoid SettingWithCopyWarning
y = df_enc['Churn']
print(f'Features: {len(feature_cols)} | Rows: {len(X):,}')

Features: 45 | Rows: 7,043


## Step 3 — Train on Full Dataset

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

scaler   = StandardScaler()

# Ensure 'tenure_segment' is one-hot encoded before scaling
# It was missed by the previous get_dummies as it was Categorical dtype.
# Use isinstance(dtype, pd.CategoricalDtype) for future-proofing, as pd.api.types.is_categorical_dtype is deprecated.
if 'tenure_segment' in X.columns and isinstance(X['tenure_segment'].dtype, pd.CategoricalDtype):
    X['tenure_segment'] = X['tenure_segment'].astype(str)

# Apply one-hot encoding to all object columns in X, which should now include 'tenure_segment'
X = pd.get_dummies(X, drop_first=True)

# Update feature_cols to reflect the new columns after one-hot encoding
feature_cols = X.columns.tolist()

# Fit scaler AND transform in one step — correct for full-dataset training
X_scaled = scaler.fit_transform(X)
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=2,
                             random_state=42, class_weight='balanced')
# Train on the FULL dataset — we use all data for deployment
rf.fit(X_scaled, y)
print(f'✅ Trained on {len(X):,} customers using {len(feature_cols)} features.')

✅ Trained on 7,043 customers using 47 features.


## Step 4 — Save Pipeline (model + scaler + feature_cols together)

In [15]:
# Bundle model, scaler, and feature list together — they must be used as a set
pipeline = {'model': rf, 'scaler': scaler, 'feature_cols': feature_cols}
# wb = write binary — pickle uses bytes, not text
with open('kova_dashboard/model_pipeline.pkl', 'wb') as f:
    # Serialise the pipeline dictionary to the file
    pickle.dump(pipeline, f)
print('✅ kova_dashboard/model_pipeline.pkl saved')

✅ kova_dashboard/model_pipeline.pkl saved


## Step 5 — Score All Customers and Save

In [16]:
# Get churn probability for every customer
y_proba_all = rf.predict_proba(X_scaled)[:,1]
df_display = df_prep.copy()
df_display['customerID']       = customer_ids.values
df_display['churn_probability'] = y_proba_all
df_display['risk_score']        = (y_proba_all*100).round(1)
df_display['risk_level']        = pd.cut(y_proba_all, bins=[0,.3,.6,1.], labels=['Low','Medium','High'])

display_cols = ['customerID','Contract','tenure','MonthlyCharges','TotalCharges',
                'num_services','tenure_segment','churn_probability','risk_score','risk_level','Churn']
display_cols = [c for c in display_cols if c in df_display.columns]
# Save scored customers for the dashboard to load
df_display[display_cols].to_csv('kova_dashboard/scored_customers.csv', index=False)
print('✅ kova_dashboard/scored_customers.csv saved')
print(f"  High:   {(df_display['risk_level']=='High').sum():,}")
print(f"  Medium: {(df_display['risk_level']=='Medium').sum():,}")
print(f"  Low:    {(df_display['risk_level']=='Low').sum():,}")

✅ kova_dashboard/scored_customers.csv saved
  High:   1,730
  Medium: 1,178
  Low:    4,106


## Step 6 — Verify

In [17]:
import os
for f in ['kova_dashboard/model_pipeline.pkl','kova_dashboard/scored_customers.csv']:
    print(f'✅ {f} ({os.path.getsize(f)//1024} KB)')
print()
print('Next steps:')
print('1. Save utils.py to kova_dashboard/')
print('2. Save app.py  to kova_dashboard/')
print('3. cd kova_dashboard && streamlit run app.py')

✅ kova_dashboard/model_pipeline.pkl (28286 KB)
✅ kova_dashboard/scored_customers.csv (541 KB)

Next steps:
1. Save utils.py to kova_dashboard/
2. Save app.py  to kova_dashboard/
3. cd kova_dashboard && streamlit run app.py


## utils.py — Save This File to kova_dashboard/utils.py

```python
import pandas as pd
import numpy as np
import pickle

OPTIMAL_THRESHOLD = 0.35

def load_pipeline(path='model_pipeline.pkl'):
    with open(path, 'rb') as f:
        return pickle.load(f)

def score_customers(df_raw, pipeline):
    from sklearn.preprocessing import LabelEncoder
    df = df_raw.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)  # new customers have zero total charges
    if 'Churn' in df.columns:
        if df['Churn'].dtype == object:
            le = LabelEncoder()
            df['Churn'] = le.fit_transform(df['Churn'].astype(str))
    else:
        df['Churn'] = -1
    df['tenure_charge_ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
    sc = [c for c in ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                       'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
          if c in df.columns]
    for col in sc:
        df[f'{col}_binary'] = df[col].apply(lambda x: 1 if str(x).lower()=='yes' else 0)
    df['num_services'] = df[[f'{c}_binary' for c in sc]].sum(axis=1)
    df['contract_numeric']  = df['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})
    df['paperless_numeric'] = (df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction'] = df['contract_numeric'] * df['paperless_numeric']
    df['tenure_segment'] = pd.cut(df['tenure'], bins=[0,12,36,float('inf')],
                                   labels=['New','Established','Loyal'])
    cat_cols = [c for c in df.select_dtypes(include='object').columns
                if c not in ['Churn','customerID']]
    df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df_enc = df_enc.reindex(columns=pipeline['feature_cols'], fill_value=0)
    X_s    = pipeline['scaler'].transform(df_enc)
    prob   = pipeline['model'].predict_proba(X_s)[:,1]
    out = df_raw.copy()
    out['churn_probability']    = prob
    out['risk_score']           = (prob*100).round(1)
    out['risk_level']           = pd.cut(prob,bins=[0,.3,.6,1.],labels=['Low','Medium','High'])
    out['will_churn_prediction'] = (prob >= OPTIMAL_THRESHOLD).astype(int)
    return out.sort_values('churn_probability', ascending=False)
```


## app.py — Save This File to kova_dashboard/app.py

```python
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import io

st.set_page_config(page_title="Kova Analytics — Churn Risk", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("scored_customers.csv")

df_all = load_data()
st.title("🏪 Kova Analytics — Customer Churn Risk Dashboard")
st.markdown("**Pilot Programme — Month 1 of 3**")
st.markdown("---")

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric("Total Customers", f"{len(df_all):,}")
with c2:
    high = (df_all['risk_level']=='High').sum()
    st.metric("High Risk", f"{high:,}", help="Churn probability > 60%")
with c3:
    med = (df_all['risk_level']=='Medium').sum()
    st.metric("Medium Risk", f"{med:,}")
with c4:
    st.metric("Recommended Offers", f"{min(500,high):,}", help="Up to 500/month")

st.sidebar.title("Filters")
contract_opts = ['All']+sorted(df_all['Contract'].unique().tolist()) if 'Contract' in df_all.columns else ['All']
sel_contract  = st.sidebar.selectbox("Contract Type", contract_opts, index=0)
sel_risk      = st.sidebar.selectbox("Risk Level", ['All','High','Medium','Low'], index=0)
n_display     = st.sidebar.slider("Customers to display", 10, 500, 100, 10)
st.sidebar.markdown("---")
st.sidebar.caption("Model: Random Forest | Threshold: 0.35\nCV Recall: 74% ±4% | Fairness Audit: Week 9\nNext Review: Week 4 Fairness Audit\nBranch filter: available upon data provision")

df_f = df_all.copy()
if sel_contract != 'All': df_f = df_f[df_f['Contract']==sel_contract]
if sel_risk     != 'All': df_f = df_f[df_f['risk_level']==sel_risk]
df_f = df_f.sort_values('churn_probability', ascending=False)

st.subheader(f"At-Risk Customers — Top {n_display} Shown")
st.caption(f"Showing {len(df_f.head(n_display)):,} of {len(df_f):,} filtered | {len(df_all):,} total")
cols_map = {k:v for k,v in {'customerID':'Customer ID','Contract':'Contract','tenure':'Tenure (mo)',
    'MonthlyCharges':'Monthly ($)','num_services':'Services',
    'risk_score':'Risk Score','risk_level':'Risk Level'}.items() if k in df_f.columns}
st.dataframe(df_f.head(n_display)[list(cols_map)].rename(columns=cols_map),
             use_container_width=True, height=350)

st.subheader("Export for Retention Campaign")
top500 = df_f.head(500)
export_cols = [c for c in ['customerID','Contract','tenure','MonthlyCharges','risk_score','risk_level'] if c in top500.columns]
buf = io.StringIO(); top500[export_cols].to_csv(buf, index=False)
st.download_button(f"⬇️  Export Top {min(500,len(df_f)):,} Customers (CSV)",
                   data=buf.getvalue(), file_name="kova_retention_list.csv", mime="text/csv")
st.caption("**Columns:** Customer ID, Contract, Tenure, Monthly Charges, Risk Score, Risk Level.")

st.markdown("---"); st.subheader("Risk Overview")
c1,c2,c3 = st.columns(3)
with c1:
    st.markdown("**Risk Score Distribution**")
    fig,ax=plt.subplots(figsize=(4,3))
    ax.hist(df_f['churn_probability'],bins=30,color='#2E75B6',edgecolor='white')
    ax.axvline(0.35,color='#C0392B',linestyle='--',linewidth=1.5,label='Threshold')
    ax.set_xlabel('Churn Probability'); ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()
with c2:
    if 'Contract' in df_f.columns:
        st.markdown("**Risk by Contract Type**")
        cr=df_f.groupby(['Contract','risk_level'],observed=True).size().unstack(fill_value=0)
        fig,ax=plt.subplots(figsize=(4,3))
        cr.plot(kind='bar',ax=ax,stacked=True,color=['#2E75B6','#E67E22','#C0392B'],edgecolor='white')
        ax.set_xlabel(''); ax.legend(title='Risk',fontsize=7,title_fontsize=7)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.xticks(rotation=20,ha='right',fontsize=8); plt.tight_layout(); st.pyplot(fig); plt.close()
with c3:
    if 'tenure_segment' in df_f.columns:
        st.markdown("**Avg Risk by Tenure**")
        sr=(df_f.groupby('tenure_segment',observed=True)['churn_probability'].mean()*100).sort_values(ascending=False)
        fig,ax=plt.subplots(figsize=(4,3))
        clrs=['#C0392B' if v==sr.max() else '#2E75B6' for v in sr.values]
        bars=ax.bar(sr.index,sr.values,color=clrs,edgecolor='white',width=0.5)
        for bar,val in zip(bars,sr.values):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f'{val:.1f}%',ha='center',va='bottom',fontsize=8)
        ax.yaxis.set_visible(False)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.tight_layout(); st.pyplot(fig); plt.close()

st.markdown("---")
st.caption("Kova Analytics | Churn Risk Pilot | Model: Week 7 RF | Next Review: Week 4 Fairness Audit")
```

**To run:** `cd kova_dashboard && streamlit run app.py`


---
## ✅ Week 10 Complete
**Next:** `week11/*_STARTER.ipynb`

---
*github.com/InternshipInABook*